# Exploración de PDB - familia proteíca "Isomerasas" 

## Objetivo
Desarrollar un análisis exploratorio de las estructuras de una familia de proteínas depositadas en el PDB,
utilizando consultas a su API y herramientas básicas de programación y visualización.

### Parte 1: Consulta de estructuras en el PDB

Realiza una búsqueda de las estructuras correspondientes a la familia de proteínas asignada a tu equipo.
Puedes utilizar directamente la PDB REST API mediante solicitudes POST con parámetros en formato JSON, o
utilizar pypdb como alternativa.

Desarrolla un script que permita recuperar los identificadores de las estructuras que cumplen con tu criterio de búsqueda. Documenta brevemente cómo definiste la consulta y qué consideraste como una estructura
perteneciente a tu familia.



In [1]:
# Definición de variables y cargar librerias
import requests

url = "https://search.rcsb.org/rcsbsearch/v2/query"
pfam_ID = "PF00697" #código de la familia de proteinas
n_familia = "phosphoribosylanthranilate isomerase" #nombre de la familia

In [2]:
# Consulta - estamos buscando todas las estructuras de PDB que contengan
# una proteína de la familia PRAI con el código pfam
def consulta_pdb(query_node, return_type="entry"):
    payload = {
        "query": query_node,
        "return_type": return_type,
        "request_options": {"return_all_hits": True},
    }
    r = requests.post(url, json=payload, timeout=60)
    if r.status_code == 204:
        return []
    r.raise_for_status()
    data = r.json()
    print(f"total_count reportado por el API: {data.get('total_count')}") # numero de resultados por el API
    return [hit["identifier"] for hit in data["result_set"]]

In [3]:
# Busqueda por pfam
query_pfam = {
    "type": "group",
    "logical_operator": "and",
    "nodes": [
        {"type": "terminal", "service": "text",
         # primera condicion - que el tipo sea pfam
         "parameters": {"attribute": "rcsb_polymer_entity_annotation.type",
                        "operator": "exact_match", "value": "Pfam"}}, 
        {"type": "terminal", "service": "text",
         # segunda condicion - que el codigo de anotacion sea el que pfam_ID
         "parameters": {"attribute": "rcsb_polymer_entity_annotation.annotation_id",
                        "operator": "exact_match", "value": pfam_ID}},
    ],
}

ids_pfam = consulta_pdb(query_pfam) #se consulta y guarda la lista de IDs
print(f"Entradas por pfam ({pfam_ID}): {len(ids_pfam)}")


total_count reportado por el API: 11
Entradas por pfam (PF00697): 11


In [4]:
# Busqueda por familia
# Se busca el nombre de la familia en cualquier texto y es más amplia que pfam
query_texto = {
    "type": "terminal",
    "service": "full_text",
    "parameters": {"value": n_familia},
}
ids_texto = consulta_pdb(query_texto)
print(f"Entradas por texto '{n_familia}': {len(ids_texto)}")

total_count reportado por el API: 34
Entradas por texto 'phosphoribosylanthranilate isomerase': 34


In [4]:
# Busqueda por familia
# Se busca el nombre de la familia en cualquier texto y es más amplia que pfam
query_texto = {
    "type": "terminal",
    "service": "full_text",
    "parameters": {"value": n_familia},
}
ids_texto = consulta_pdb(query_texto)
print(f"Entradas por texto '{n_familia}': {len(ids_texto)}")

total_count reportado por el API: 34
Entradas por texto 'phosphoribosylanthranilate isomerase': 34


In [5]:
# Comparación del numero reportado por la busqueda por pfam y por el nombre de la familia
set_pfam, set_texto = set(ids_pfam), set(ids_texto)
print("Resultados por pfam (IDs):", len(set_pfam - set_texto))
print("Resultados por busqueda de texto (IDs):", len(set_texto - set_pfam))
print("Cuales estan en ambos:", len(set_pfam & set_texto))

ids_final = sorted(set_pfam) # guardamos las de pfam por que a veces el nombre no encuentra
# todas las estructuras y en muchos casos suelen ser falsos positivos
with open("ids_familia.txt", "w") as f:
    f.write("\n".join(ids_final))
print(f"Guardados {len(ids_final)} IDs en ids_familia.txt") # guardamos los IDs

Resultados por pfam (IDs): 0
Resultados por busqueda de texto (IDs): 23
Cuales estan en ambos: 11
Guardados 11 IDs en ids_familia.txt


#### Interpretación

Se uso la base de datos PDB, usamos primeramente el crítero Pfam (PF00697) que contiene familias y dominios de proteínas clasificándolos en base a su secuencia, en este caso lo que hicimos fue buscar todas aquellas entradas con al menos una entidad polimética anotada con ese dominio. 

Seguidamente usamos la busqueda por texto del nombre de familia proteíca en este caso usamos: "phosphoribosylanthranilate isomerase". Con esto finalmente solo nos quedamos con los ID de las proteínas que se clasificaron en base al Pfam, por que la anotación se da en base en la secuencias, mientras que la busqueda por texto dado que muchas veces cuando se asignan nombres a las familias proteícas pueden haber indiscrepacias que limiten la búsqueda. Finalmente nos quedamos con 11 IDs que se guardaron en un archivo txt. 


### Parte 03. Exploración estructural
Selecciona al menos una estructura representativa de tu familia y explórala utilizando Mol*, PyMOL o  ChimeraX. Incluye una imagen de la estructura y señala, cuando sea posible, elementos de estructura  secundaria, la organización general del plegamiento y algún ligando asociado si está presente en el registro. 
Acompaña la imagen con el identificador PDB y una breve descripción de lo que estás mostrando. 

La siguiente imagen muestra la estructura proteica de los monómeros de la enzima fosforribosil antranilato isomerasa dimérica de Thermotoga maritima. El identificador de esta proteína es ***1DL3*** en la base de datos de PDB, las imágenes fueron generadas utilizando la base de datos Mol* y realizando la búsqueda de la proteína con el mismo identificador que PDB. 

![protein](images/1DL3-protein.png)

La estructura presenta α-hélices en magenta, hebras β en dorado, hélices 3₁₀ en morado y loops en gris/verde. Las hebras β forman un barril central rodeado por hélices α, el plegamiento de barril TIM (βα)₈. En el borde del barril se observa un ligando en ball&stick, cerca del sitio activo, y moléculas de agua como esferas rosadas.
Por otro lado, se observa un ion sulfato, SO4 (estructura de molécula ball-stick), unido cerca del extremo de las hebras β del barril. Probablemente proviene de la solución de cristalización y ocupa el sitio de unión del grupo fosfato del sustrato, lo que sugiere la ubicación del sitio activo. La base de datos de PDB indica que la proteína tiene una longitud de ~385 a.a. 

![secondary-structure](images/1DL3-secondary-structure.png)

### Parte 04. Interpretación de los resultados
A partir de tus resultados, ¿qué puedes decir sobre la representación de tu familia en el PDB? 

***- ¿Qué observaciones te parecen más interesantes sobre los organismos, los métodos experimentales o la  distribución temporal de las estructuras?***

***- ¿Qué limitaciones tiene este análisis? Considera, por ejemplo, que el número de estructuras depositadas  no necesariamente representa la diversidad real de la familia en la naturaleza.***

La principal limitación es que el número de estructuras depositadas en el PDB no representa la diversidad real de la familia en la naturaleza. Favorece organismos modelo, patógenos y termófilos cuyas proteínas son estables y fáciles de cristalizar, por lo que en la lista predominan E. coli, Thermotoga maritima, Thermococcus kodakarensis, Pyrococcus furiosus y Corynebacterium glutamicum, mientras que casi no hay eucariotas ni la mayoría de los filos bacterianos. 

Además la base de datos presenta un poco de redundancia, ya que varias entradas son la misma proteína con mutantes o ligandos distintos (tres de T. maritima y dos de T. kodakarensis), de modo que los 11 IDs equivalen a unas 6 especies (aproximadamente), y las estructuras no son unidades independientes. También hay heterogeneidad entre las entradas, pues se mezclan proteínas completas, un fragmento truncado (2KZH), variantes de ingeniería (1DL3) y enzimas bifuncionales donde la PRAI es solo un dominio (1PII, 7ETY). Además, el filtro por familia condiciona las estructuras a analizar. 

Finalmente, el conteo de estructuras depende de fuentes secundarias y posiblemente desactualizadas, para una versión más sólida habría que agrupar por identidad de secuencia, contar clústeres en lugar de entradas y contrastar con UniProt, Pfam o InterPro, que sí cubren secuencias sin estructura.